In [0]:
%run ../07_Common/00_setup

In [0]:
df_silver_users = spark.table("workspace.silver.users")

df_dim_users = (
    df_silver_users
    .select(
        F.col("id").alias("user_id"),
        F.col("first_name"),
        F.col("last_name"),
        F.col("gender"),
        F.col("age").cast("int").alias("age"),
        F.col("email"),
        F.col("address_city").alias("city"),
        F.col("address_state").alias("state"),
        F.col("address_country").alias("country"),
    )
)

print(f"📊 Registros en dim_users: {df_dim_users.count()}")

In [0]:
try:
    (
        df_dim_users.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable("workspace.gold.dim_users")
    )
    estado_final = "success"
    print("Escritura exitosa en gold.dim_users")
except Exception as e:
    estado_final = "failed"
    print(f"❌ ERROR: {e}")

spark.sql(f"""
    UPDATE workspace.control.gold_aggregation_config
    SET last_run_status = '{estado_final}'
    WHERE entity_name = 'dim_users'
""")
dbutils.notebook.exit(f"{estado_final.upper()} | dim_users")